## Executive Summary

**Purpose:** One call from the requirements (ratio, torque, speed, life, envelope, input shaft) to a complete, checked single-stage planetary design and the files needed to make it.

**What it does:** `design_planetary` searches every tooth-count set near the target ratio, for each allowed number of planets and each standard module that fits between the sun bore and the envelope, and finds the narrowest face width at which every check passes: mesh geometry, ISO 6336 pitting and bending, planet bearing life, pin, carrier, key, ring fixing and axial length. It returns the passing designs ranked. `PlanetaryDesign.write_package` writes, for the chosen design:

| File | Content |
|---|---|
| `report.md` | requirements, chosen design, every check with its value and limit, assumptions |
| `gear_data_sun.csv`, `gear_data_planet.csv`, `gear_data_ring.csv` | ISO 1340 / DIN 3966-1 gear data tables for the drawings |
| `parts.csv` | parts list with quantities, key dimensions and fits |
| `component_dimensions.csv` | dimensions and limits of pin, bore, carrier, key, ring fixing |
| `layout.svg` | front view and axial section of the stage |
| `sun.dxf`, `planet.dxf`, `ring.dxf` | tooth outlines at mean thickness, for prototypes and drawings |

**Inputs ND supplies:** outer envelope (diameter, optionally length) and the shaft diameter that carries the sun. The rest have defaults that are stated in the report and can all be overridden.

## How It Fits Into the Gear Design Workflow

**Upstream:** all other `gears` notebooks: `involute` → `layout` → `rating` → `components` → `manufacturing`.

**Downstream:** the drawings. A designer copies the data tables onto the gear drawings, the fits onto the part drawings, and uses the layout for the assembly drawing.

In [ ]:
#| default_exp gears.design

In [ ]:
#| export
from __future__ import annotations

import csv
import math
import os
from dataclasses import dataclass, field, asdict, replace

import numpy as np

from emachines.gears.involute import GearGeometry, standard_modules
from emachines.gears.layout import PlanetarySet, tooth_count_candidates, module_limits
from emachines.gears.rating import MATERIALS, GearMaterial, RatingInputs, PlanetaryRating, rate_planetary
from emachines.gears.components import (ComponentInputs, PlanetaryComponents, ParallelKey, rim_thicknesses,
                                        size_components, _hub_depth, BOLTS_88)
from emachines.gears.manufacturing import (backlash_allowances, gear_data_sheet, gear_outline, write_dxf,
                                           iso286_limits, fmt_fit, it_grade, DataRow)

In [ ]:
#| hide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Rectangle, FancyBboxPatch
from emachines.gears import layout as _lay
from emachines.gears.manufacturing import gear_outline   # exported module, so figures can sit ahead of the definitions

In [ ]:
#| hide
# drawing helpers for the explanatory figures (not exported)

_PS = _lay.PlanetarySet.build(18, 27, 72, m=2.0, n_planets=3, b=20)

INK, BLUE, RED, GREEN, GREY = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a", "#888888"
C_SUN, C_PLANET, C_RING, C_STEEL = "#d8e8d0", "#f6e7c8", "#dfe7f1", "#bdbdbd"

def _rot(P, a):
    c, s_ = np.cos(a), np.sin(a)
    return np.asarray(P) @ np.array([[c, s_], [-s_, c]])

def _label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))

def _dim(ax, p0, p1, text="", off=(0, 0), color=BLUE, fs=9, ha="center"):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->", color=color, lw=0.8, shrinkA=0, shrinkB=0))
    if text:
        ax.text((p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1], text, color=color, fontsize=fs,
                ha=ha, va="center", bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))

def _set_drawing(ps, ax, turn=np.pi / 2):
    "Draw sun, planets and ring of a PlanetarySet in mesh, turned so planet 1 is at angle `turn`."
    rho = np.pi * ps.planet.z / ps.ring.z
    ax.add_patch(Polygon(_rot(gear_outline(ps.ring), rho + turn), closed=True, fc="white", ec=INK, lw=0.7))
    for k in range(ps.n_planets):
        th = 2 * np.pi * k / ps.n_planets
        psi = th * (1 + ps.sun.z / ps.planet.z) + np.pi - np.pi / ps.planet.z
        c = ps.a_w * np.array([np.cos(th), np.sin(th)])
        ax.add_patch(Polygon(_rot(_rot(gear_outline(ps.planet), psi) + c, turn), closed=True, fc=C_PLANET, ec=INK, lw=0.7))
    ax.add_patch(Polygon(_rot(gear_outline(ps.sun), turn), closed=True, fc=C_SUN, ec=INK, lw=0.7))

## Requirements: Theory

The stage has the ring fixed in the housing, the sun on the input shaft and the carrier as output. What it must do is given by the ratio $i$, the output torque $T_{out}$, the input speed $n_{in}$ and the life $L_h$; where it must fit by the outer envelope diameter $D_{env}$ (and optionally the length $L_{env}$), the input shaft diameter $d_{sh}$ in the sun bore and, if the ring sits in a housing, the housing wall $t_h$ outside it:

In [ ]:
#| echo: false
def _requirements_figure():
    ps = _PS
    D, L, d_sh, t_h = 170.0, 50.0, 16.0, 6.0
    R = D / 2
    fig, axs = plt.subplots(1, 2, figsize=(12, 5.6), gridspec_kw=dict(width_ratios=[1.05, 1]))
    ax = axs[0]
    ax.add_patch(Circle((0, 0), R, fc=C_STEEL, ec=INK, lw=0.9))
    ax.add_patch(Circle((0, 0), R - t_h, fc=C_RING, ec=INK, lw=0.7))
    _set_drawing(ps, ax)
    ax.add_patch(Circle((0, 0), d_sh / 2, fc="#999", ec=INK, lw=0.7))
    ax.add_patch(Circle((0, 0), R, fill=False, ec=RED, lw=1.2, ls="--"))
    _dim(ax, (-R, -R - 8), (R, -R - 8), "envelope_d  $D_{env}$", color=RED)
    _label(ax, "housing_wall $t_h$ (0: the ring is the outer wall)", (R * np.cos(0.6) - 2, R * np.sin(0.6) - 2), (40, R + 14), color=INK)
    _label(ax, "shaft_d $d_{sh}$ (input shaft in the sun bore)", (d_sh / 2 * 0.7, -d_sh / 2 * 0.7), (25, -R - 2), color=INK)
    ax.set_xlim(-R - 8, R + 70); ax.set_ylim(-R - 18, R + 22); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("front view", fontsize=10)
    ax = axs[1]
    b, tc = 20.0, 5.0
    x0 = 0.0
    ax.add_patch(Rectangle((x0, -R), L, 2 * R, fill=False, ec=RED, lw=1.2, ls="--"))
    w_r = b + 2 * tc + 8
    for sg in (1, -1):
        y0 = ps.ring.da / 2 if sg > 0 else -(R - t_h)
        ax.add_patch(Rectangle((x0 + 3, y0), w_r, R - t_h - ps.ring.da / 2, fc=C_RING, ec=INK, lw=0.7))
        y1 = R - t_h if sg > 0 else -R
        ax.add_patch(Rectangle((x0, y1), L, t_h, fc=C_STEEL, ec=INK, lw=0.7))
    xg = x0 + 3 + tc + 4
    for s in (1, -1):
        ax.add_patch(Rectangle((xg, s * ps.a_w - ps.planet.da / 2), b, ps.planet.da, fc=C_PLANET, ec=INK, lw=0.7))
    ax.add_patch(Rectangle((xg, -ps.sun.da / 2), b, ps.sun.da, fc=C_SUN, ec=INK, lw=0.7))
    for xp in (x0 + 3 + 1, xg + b + 2):
        ax.add_patch(Rectangle((xp, ps.sun.da / 2 + 2), tc, ps.a_w + 12 - ps.sun.da / 2 - 2, fc="#cfe8d6", ec=GREEN, lw=0.7))
        ax.add_patch(Rectangle((xp, -ps.a_w - 12), tc, ps.a_w + 12 - ps.sun.da / 2 - 2, fc="#cfe8d6", ec=GREEN, lw=0.7))
    ax.add_patch(Rectangle((x0 - 25, -d_sh / 2), xg + b + 25, d_sh, fc="#999", ec=INK, lw=0.7))
    ax.add_patch(Rectangle((xg + b + 2 + tc, -10), L + 20 - (xg + b + 2 + tc), 20, fc="#cfe8d6", ec=GREEN, lw=0.7))
    ax.annotate("", xy=(x0 - 4, 0), xytext=(x0 - 24, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=1.4))
    ax.text(x0 - 26, 12, "input: sun\n$n_{in}$", ha="left", va="bottom", fontsize=9, color=BLUE)
    ax.annotate("", xy=(L + 42, 0), xytext=(L + 22, 0), arrowprops=dict(arrowstyle="-|>", color=GREEN, lw=1.4))
    ax.text(L + 22, 18, "output: carrier\n$T_{out}$, $n_{in}/i$", ha="left", va="bottom", fontsize=9, color=GREEN)
    ax.text(x0 + L / 2, R + 3, "ring fixed in the housing", ha="center", fontsize=9)
    _dim(ax, (x0, -R - 8), (x0 + L, -R - 8), "envelope_length $L_{env}$", color=RED)
    ax.set_xlim(-30, L + 48); ax.set_ylim(-R - 18, R + 22); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("axial section", fontsize=10)
    plt.tight_layout()
_requirements_figure()

With the ring fixed the ratio and the output speed are

$$i = 1 + \frac{z_r}{z_s},\qquad n_{out} = \frac{n_{in}}{i},$$

and a tooth-count set is accepted when its relative ratio error is within the tolerance,

$$\varepsilon_i = \frac{i - i_{target}}{i_{target}},\qquad |\varepsilon_i| \le \mathrm{ratio\_tol}\ (\text{default } 3\,\%).$$

The gears are rated at $T_{out}$ on the carrier, multiplied by the application factor $K_A$ (default 1.25) inside the rating. Every other input has a default (materials, flank tolerance classes, finish, safety factors $S_{H,min} = 1.0$ and $S_{F,min} = 1.4$, planet counts 3 to 5, sun teeth 12 to 40); all of them are written into the report.

The sun–shaft joint is chosen with `hub_joint` (`key`, `spline`, `press`, `adhesive` or `integral`, see `gears.components`), with its details in `joint_options`. A joint without backlash fixes the sun radially, so it cannot be combined with `floating_sun = True`; the search refuses that combination.

The carrier is `double` (two plates joined by posts between the planets, the default) or `single` (one plate with cantilevered pins, needed at high ratios where the planets leave no room for posts). `m_min` sets the smallest module the search may use; without it the search, which ranks by length and margin, can pick modules too fine to make.

In [ ]:
#| export
@dataclass(frozen=True)
class PlanetaryRequirements:
    """
    What the stage must do and where it must fit. Lengths mm, torque N·m, speed rpm.

    ratio, torque_out, n_in, life_h : target ratio, output torque for rating, input speed, required life [h]
    envelope_d, envelope_length     : outer diameter and axial length available (length optional)
    shaft_d                         : input shaft diameter in the sun bore
    hub_joint                       : 'key', 'spline', 'press', 'adhesive' or 'integral' (see gears.components)
    carrier                         : 'double' (posts between the planets) or 'single' (one plate, cantilevered pins)
    m_min                           : smallest module to consider [mm] (0: no limit)
    joint_options                   : further ComponentInputs fields for the joint, e.g. {'spline_fit': 'tight'}
                                      or {'press_fit': 'H6/s5'}
    ratio_tol                       : allowed relative ratio error
    n_planets                       : planet counts to try
    K_A                             : application factor
    materials                       : {'sun'|'planet'|'ring': key of gears.rating.MATERIALS}
    grade, ring_grade               : ISO 1328-1 flank tolerance class of sun/planets and of the ring
    flank_finish                    : 'ground' or 'hobbed' for sun and planets
    S_Hmin, S_Fmin                  : required safety factors
    floating_sun                    : the sun is radially free (load sharing)
    housing_wall                    : housing wall outside the ring (0: ring is the outer wall)
    series_ii                       : also allow ISO 54 series II modules
    zs_range                        : sun tooth counts to search
    """
    ratio: float
    torque_out: float
    n_in: float
    envelope_d: float
    shaft_d: float
    envelope_length: float | None = None
    life_h: float = 10_000.0
    hub_joint: str = "key"
    carrier: str = "double"
    m_min: float = 0.0
    joint_options: dict = field(default_factory=dict)
    ratio_tol: float = 0.03
    n_planets: tuple[int, ...] = (3, 4, 5)
    K_A: float = 1.25
    materials: dict = field(default_factory=lambda: {"sun": "case_carburized", "planet": "case_carburized",
                                                     "ring": "qt_nitrided"})
    grade: int = 6
    ring_grade: int = 7
    flank_finish: str = "ground"
    S_Hmin: float = 1.0
    S_Fmin: float = 1.4
    floating_sun: bool = False
    housing_wall: float = 0.0
    series_ii: bool = True
    zs_range: tuple[int, int] = (12, 40)

    def rating_inputs(self) -> RatingInputs:
        return RatingInputs(torque_out=self.torque_out, n_in=self.n_in, life_h=self.life_h, K_A=self.K_A,
                            grade=self.grade, floating_sun=self.floating_sun, S_Hmin=self.S_Hmin,
                            S_Fmin=self.S_Fmin, flank_finish=self.flank_finish)

    def component_inputs(self) -> ComponentInputs:
        return ComponentInputs(envelope_d=self.envelope_d, shaft_d=self.shaft_d,
                               envelope_length=self.envelope_length, hub_joint=self.hub_joint,
                               housing_wall=self.housing_wall, carrier=self.carrier, **self.joint_options)

    def material_set(self) -> dict[str, GearMaterial]:
        return {k: MATERIALS[v] for k, v in self.materials.items()}

## The Search: Theory

For every number of planets $N$ and every tooth-count set within the ratio tolerance (`tooth_count_candidates`):

1. **Module window.** `module_limits`, raised to at least `m_min`, with the hard ISO 6336-3 rim limits (sun rim ≥ 0.5 tooth depths over the key groove, ring rim ≥ 1.75 m). Thinner-than-ideal rims are allowed and charged through $Y_B$. Every standard module in the window is tried.
2. **Geometry.** `PlanetarySet.build`; sets that fail a mesh, neighbour or tip-thickness check are dropped.
3. **Face width.** Starting from

   $$b_{lo} = \max(5,\ \lceil 6m \rceil)\ \text{mm}$$

   (narrower teeth are hard to hold square and to inspect) and growing in 1 mm steps up to

   $$b_{hi} = \min\big(1.2\,d_s,\ L_{env} - n_c t_c - 2s_w - 2c_{ax}\big),\qquad n_c = 2\ (\text{double}) \text{ or } 1\ (\text{single}),\ t_c = 4,\ s_w = 1,\ c_{ax} = 0.5\ \text{mm (first estimate of the axial stack)},$$

   each width is evaluated: the set is rated with rims from a first component sizing, the components are sized, and both are repeated once so that the planet rim matches the bearing bore actually chosen. The first width at which everything passes is kept. Gears are only as wide as they need to be; extra width costs length and makes the load less even across the face. If the geometry is invalid at some width, that module is abandoned.
4. **Ranking.** With the safety margin

   $$\mu = \min\!\Big(\frac{S_H}{S_{H,min}},\ \frac{S_F}{S_{F,min}}\Big),$$

   passing designs are sorted by the key $\big(\mathrm{round}(|\varepsilon_i| / 0.005),\ \mathrm{round}(L_{stack}),\ -\mu\big)$: ratio error in 0.5 % steps, then axial length to the millimetre, then the larger margin first.

The search typically takes a few seconds and evaluates a few hundred designs.

In [ ]:
#| echo: false
def _search_figure():
    fig, ax = plt.subplots(figsize=(11, 7.4))
    def box(x, y, text, w=5.6, h=0.78, fc="white", ec=INK):
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.06", fc=fc, ec=ec, lw=0.9))
        ax.text(x, y, text, ha="center", va="center", fontsize=9)
    def arrow(p0, p1, color=INK, text="", tx=(0, 0), conn="arc3"):
        ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="-|>", color=color, lw=0.9, connectionstyle=conn))
        if text:
            ax.text((p0[0] + p1[0]) / 2 + tx[0], (p0[1] + p1[1]) / 2 + tx[1], text, fontsize=8.5, color=color, ha="center", va="center")
    x = 4.0
    ys = [9.0, 7.9, 6.8, 5.7, 4.6, 3.5, 2.4]
    texts = ["requirements: $i$, $T_{out}$, $n_{in}$, $L_{h}$, $D_{env}$, $L_{env}$, $d_{sh}$",
             "for each number of planets $N$",
             "for each tooth-count set $(z_s, z_p, z_r)$\n$|i - i_{target}|\\,/\\,i_{target} \\leq$ ratio_tol, assembly and neighbour conditions",
             "module window $[m_{lo}, m_{hi}]$ from bore and envelope;\nfor each standard module $m$ in it",
             "$b = \\max(5,\\ \\lceil 6m \\rceil)$ mm",
             "evaluate: build set → rate (rims from first sizing) →\nsize components → rate again → size again",
             "rating ok and components ok?"]
    fills = ["#eef3fa", "white", "white", "white", "white", "#fdf6e6", "white"]
    for y, t, f in zip(ys, texts, fills):
        box(x, y, t, fc=f, h=0.78 if "\n" not in t else 0.9)
    for y0, y1 in zip(ys[:-1], ys[1:]):
        arrow((x, y0 - 0.45), (x, y1 + 0.47))
    # yes → keep
    box(x, 1.1, "keep this design (narrowest passing $b$); next module", fc="#e8f4ea", ec=GREEN)
    arrow((x, ys[-1] - 0.42), (x, 1.1 + 0.42), color=GREEN, text="yes", tx=(0.35, 0))
    # no → widen
    box(9.6, 3.5, "$b \\leftarrow b + 1$ mm\nwhile $b \\leq b_{hi}$", w=3.0, h=0.9, fc="#fbeceb", ec=RED)
    arrow((x + 2.85, ys[-1]), (9.6, 3.5 - 0.5), color=RED, text="no", tx=(0.4, -0.15), conn="angle,angleA=0,angleB=90")
    arrow((9.6 - 1.55, 3.5), (x + 2.85, 3.5), color=RED, text="evaluate again", tx=(0, 0.18))
    ax.text(9.6, 2.15, "geometry invalid or $b > b_{hi}$:\nno design for this $m$", fontsize=8.5, ha="center", va="top", color=RED)
    # rank
    box(x, -0.25, "after all loops, rank the kept designs by: 1. ratio error in 0.5 % steps,\n2. axial length rounded to mm, 3. safety margin (larger first)",
        w=7.0, h=0.9, fc="#eef3fa")
    ax.annotate("", xy=(x, -0.25 + 0.5), xytext=(x, 1.1 - 0.42), arrowprops=dict(arrowstyle="-|>", color=INK, lw=0.9, ls="--"))
    ax.set_xlim(0.3, 11.3); ax.set_ylim(-0.9, 9.6); ax.axis("off")
    ax.set_title("The design search", fontsize=10)
    plt.tight_layout()
_search_figure()

In [ ]:
#| export
@dataclass
class PlanetaryDesign:
    "A complete checked design: requirements, gear set, rating, components and backlash allowances."
    req: PlanetaryRequirements
    gears: PlanetarySet
    rating: PlanetaryRating
    components: PlanetaryComponents
    allowances: dict

    @property
    def margin(self) -> float:
        "Smallest ratio of achieved to required safety factor."
        return min(self.rating.S_H / self.req.S_Hmin, self.rating.S_F / self.req.S_Fmin)

    @property
    def ratio_error(self) -> float:
        return (self.gears.ratio - self.req.ratio) / self.req.ratio

    @property
    def length(self) -> float:
        return self.components.axial["stack"]

    def label(self) -> str:
        g = self.gears
        return (f"{g.sun.z}/{g.planet.z}/{g.ring.z} ×{g.n_planets}, m {g.sun.m:g}, b {g.sun.b:g} mm: "
                f"i = {g.ratio:.4f} ({self.ratio_error * 100:+.2f} %), S_H {self.rating.S_H:.2f}, "
                f"S_F {self.rating.S_F:.2f}, length {self.length:.1f} mm, limited by {self.rating.limiting}")


def evaluate(req: PlanetaryRequirements, zs: int, zp: int, zr: int, m: float, n_planets: int,
             b: float) -> PlanetaryDesign | None:
    "Build, rate and size one design; None if the geometry itself is not valid."
    try:
        ps = PlanetarySet.build(zs, zp, zr, m=m, n_planets=n_planets, b=b)
    except ValueError:
        return None
    if not ps.ok:
        return None
    ci = req.component_inputs()
    inp = req.rating_inputs()
    mats = req.material_set()
    try:
        r0 = rate_planetary(ps, inp, mats, rim_thicknesses(ps, ci))
        c0 = size_components(ps, r0, ci, req.life_h, req.n_in, req.K_A)
        rims = rim_thicknesses(ps, ci, planet_bore=c0.bearing["Ew"])
        r = rate_planetary(ps, inp, mats, rims)
        c = size_components(ps, r, ci, req.life_h, req.n_in, req.K_A)
    except ValueError:
        return None
    bl = backlash_allowances(ps, req.grade)
    return PlanetaryDesign(req, ps, r, c, bl)


def design_planetary(req: PlanetaryRequirements, max_results: int = 10) -> list[PlanetaryDesign]:
    "Search tooth counts, planets, modules and face widths; return passing designs, best first (see theory)."
    ci = req.component_inputs()
    if req.floating_sun and not (ci.hub_joint == "key" or (ci.hub_joint == "spline" and ci.spline_fit == "sliding")):
        raise ValueError("a floating sun needs a key or a sliding spline; joints without backlash fix the sun radially")
    t2 = _hub_depth(ci)
    found: list[PlanetaryDesign] = []
    for N in req.n_planets:
        for c in tooth_count_candidates(req.ratio, N, req.ratio_tol, req.zs_range):
            m_lo, m_hi = module_limits(req.envelope_d, req.shaft_d if req.hub_joint != "integral" else 0.0,
                                       c.zs, c.zr, t2=t2, housing_wall=req.housing_wall,
                                       ring_rim=1.75, sun_rim=0.5)
            for m in standard_modules(max(m_lo, req.m_min), m_hi, series_ii=req.series_ii):
                b_lo = max(5.0, math.ceil(6 * m))
                b_hi = 1.2 * c.zs * m
                if req.envelope_length is not None:
                    b_hi = min(b_hi, req.envelope_length - (1 if req.carrier == "single" else 2) * 4.0 - 2 * 1.0 - 2 * 0.5)
                b = b_lo
                while b <= b_hi + 1e-9:
                    d = evaluate(req, c.zs, c.zp, c.zr, m, N, b)
                    if d is None:
                        break
                    if d.rating.ok and d.components.ok:
                        found.append(d)
                        break
                    b += 1.0
    found.sort(key=lambda d: (round(abs(d.ratio_error) / 0.005), round(d.length), -d.margin))
    return found[:max_results]

## Drawing: Front View and Axial Section

`layout_svg` draws the stage to scale: on the left the front view with true involute teeth, carrier plate outline, pin holes, the input shaft with its key, and the ring outer diameter $2R = D_{env} - 2t_h$; on the right the axial half-section with the carrier plates, gears, pins, bearings and thrust washers. Main dimensions are written beside the views. The teeth are drawn at the mean thickness, $\Delta s = (A_{sne} + A_{sni})/2$.

**Phasing.** For the teeth to mesh in the drawing, each gear is turned to the right angle. Planet $k$ sits at

$$\theta_k = \frac{2\pi k}{N},\qquad k = 0, \dots, N-1 .$$

With a sun tooth on the $+x$ axis, planet 0 (at $\theta_0 = 0$) must face the sun with a tooth *space*: its tooth 0 points away from the sun, $\pi$, turned by half a pitch, $-\pi/z_p$. Carrying a planet round the fixed sun to $\theta_k$ rolls it on the sun, which turns it by $\theta_k(1 + z_s/z_p)$, so

$$\psi_k = \theta_k\Big(1 + \frac{z_s}{z_p}\Big) + \pi - \frac{\pi}{z_p}.$$

The ring is turned by $\rho = \pi z_p / z_r$ so that a ring tooth space meets the outward tooth of planet 0; the assembly condition $(z_s + z_r)/N \in \mathbb{Z}$ (checked in `gears.layout`) makes every other planet mesh with the ring too.

In [ ]:
#| echo: false
def _phase_figure():
    ps = _PS
    zs, zp, zr, N = ps.sun.z, ps.planet.z, ps.ring.z, ps.n_planets
    fig, ax = plt.subplots(figsize=(8, 7.6))
    _set_drawing(ps, ax, turn=0.0)
    k = 1
    th = 2 * np.pi * k / N
    psi = th * (1 + zs / zp) + np.pi - np.pi / zp
    c = ps.a_w * np.array([np.cos(th), np.sin(th)])
    ax.plot([0, ps.ring.d / 2 + 6], [0, 0], color=GREY, lw=0.6, ls="-.")
    ax.plot([0, c[0] * 1.9], [0, c[1] * 1.9], color=BLUE, lw=0.7, ls="--")
    t = np.linspace(0, th, 60)
    ax.plot(22 * np.cos(t), 22 * np.sin(t), color=BLUE, lw=0.9)
    ax.text(25 * np.cos(th / 2), 25 * np.sin(th / 2), "$\\theta_k = 2\\pi k / N$", color=BLUE, fontsize=9.5)
    # planet tooth 0 direction
    u = np.array([np.cos(psi), np.sin(psi)])
    ax.plot([c[0], c[0] + 34 * u[0]], [c[1], c[1] + 34 * u[1]], color=RED, lw=1.0)
    ax.plot([c[0], c[0] + 34], [c[1], c[1]], color=GREY, lw=0.6, ls="-.")
    t = np.linspace(0, np.mod(psi, 2 * np.pi), 80)
    ax.plot(c[0] + 12 * np.cos(t), c[1] + 12 * np.sin(t), color=RED, lw=0.9)
    _label(ax, "$\\psi_k = \\theta_k\\,(1 + z_s/z_p) + \\pi - \\pi/z_p$\n(turn of planet $k$'s tooth 0)", c + 12 * np.array([np.cos(np.mod(psi, 2 * np.pi) / 2), np.sin(np.mod(psi, 2 * np.pi) / 2)]),
           (-ps.ring.d / 2 - 40, ps.ring.d / 2 + 10), color=RED)
    # ring turn
    rho = np.pi * zp / zr
    rr = ps.ring.d / 2
    ax.plot([0, (rr + 14) * np.cos(rho)], [0, (rr + 14) * np.sin(rho)], color=GREEN, lw=0.8)
    t = np.linspace(0, rho, 30)
    ax.plot((rr + 9) * np.cos(t), (rr + 9) * np.sin(t), color=GREEN, lw=0.9)
    ax.text((rr + 16) * np.cos(rho / 2) + 2, (rr + 16) * np.sin(rho / 2), "$\\rho = \\pi z_p / z_r$\n(ring turn)", color=GREEN, fontsize=9, va="center")
    ax.text(c[0], c[1] - 9, f"planet k = {k}", fontsize=8.5, ha="center", va="top")
    ax.set_xlim(-rr - 45, rr + 45); ax.set_ylim(-rr - 10, rr + 25); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Phasing the gears so the teeth mesh ({zs}/{zp}/{zr}, N = {N})", fontsize=10)
    plt.tight_layout()
_phase_figure()

In [ ]:
#| export
def _poly(P: np.ndarray, sx: float, ox: float, oy: float, style: str) -> str:
    pts = " ".join(f"{ox + x * sx:.2f},{oy - y * sx:.2f}" for x, y in P)
    return f'<polygon points="{pts}" {style}/>'


def layout_svg(d: PlanetaryDesign, width_px: int = 1100) -> str:
    "SVG text of the front view and axial section of a design (mm scale, see above)."
    g, comp = d.gears, d.components
    R = d.req.envelope_d / 2 - d.req.housing_wall
    L = comp.axial["stack"]
    view_w = 2 * R + 30 + L + 25 + 150
    sx = width_px / view_w
    H = (2 * R + 40) * sx
    ox, oy = (R + 10) * sx, (R + 20) * sx
    thin = 'fill="none" stroke="#222" stroke-width="0.8"'
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width_px}" height="{H + 60:.0f}" font-family="sans-serif" font-size="11">',
           '<rect width="100%" height="100%" fill="white"/>']
    ds = {k: (v.A_sne + v.A_sni) / 2 for k, v in d.allowances["allowances"].items()}
    # ring outer circle and teeth
    out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{R * sx:.2f}" fill="#e8eef5" stroke="#222" stroke-width="1"/>')
    rho = np.pi * g.planet.z / g.ring.z                          # ring phase that meshes with the planets
    Rr = np.array([[np.cos(rho), np.sin(rho)], [-np.sin(rho), np.cos(rho)]])
    out.append(_poly(gear_outline(g.ring, ds["ring"]) @ Rr, sx, ox, oy, 'fill="white" stroke="#222" stroke-width="0.8"'))
    # carrier plate (behind), dashed
    out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{comp.carrier["plate_d"] / 2 * sx:.2f}" fill="none" stroke="#2a6" stroke-dasharray="5,3"/>')
    # planets
    for k in range(g.n_planets):
        th = 2 * np.pi * k / g.n_planets
        psi = th * (1 + g.sun.z / g.planet.z) + np.pi - np.pi / g.planet.z   # planet phase that meshes with the sun
        c, s = np.cos(psi), np.sin(psi)
        P = gear_outline(g.planet, ds["planet"]) @ np.array([[c, s], [-s, c]])
        cx, cy = g.a_w * np.cos(th), g.a_w * np.sin(th)
        out.append(_poly(P + [cx, cy], sx, ox, oy, 'fill="#f6e7c8" stroke="#222" stroke-width="0.8"'))
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy - cy * sx:.2f}" r="{comp.bearing["Ew"] / 2 * sx:.2f}" fill="white" stroke="#222" stroke-width="0.6"/>')
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy - cy * sx:.2f}" r="{comp.pin["d"] / 2 * sx:.2f}" fill="#bbb" stroke="#222" stroke-width="0.6"/>')
    # sun, shaft, key
    out.append(_poly(gear_outline(g.sun, ds["sun"]), sx, ox, oy, 'fill="#d8e8d0" stroke="#222" stroke-width="0.8"'))
    if d.req.hub_joint == "integral":
        out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{d.req.shaft_d / 2 * sx:.2f}" fill="none" stroke="#888" stroke-dasharray="4,2"/>')
    else:
        out.append(f'<circle cx="{ox:.2f}" cy="{oy:.2f}" r="{d.req.shaft_d / 2 * sx:.2f}" fill="#bbb" stroke="#222" stroke-width="0.6"/>')
    if d.req.hub_joint == "key":
        key = ParallelKey.for_shaft(d.req.shaft_d)
        y0 = d.req.shaft_d / 2 - key.t1
        out.append(f'<rect x="{ox - key.b / 2 * sx:.2f}" y="{oy - (y0 + key.h) * sx:.2f}" width="{key.b * sx:.2f}" height="{key.h * sx:.2f}" fill="#888" stroke="#222" stroke-width="0.5"/>')
    out.append(f'<line x1="{ox - (R + 5) * sx:.1f}" y1="{oy:.1f}" x2="{ox + (R + 5) * sx:.1f}" y2="{oy:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<line x1="{ox:.1f}" y1="{oy - (R + 5) * sx:.1f}" x2="{ox:.1f}" y2="{oy + (R + 5) * sx:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="{ox:.1f}" y="{H + 15:.0f}" text-anchor="middle">Front view (carrier plate dashed)</text>')
    # axial half section, axis horizontal at the bottom of the section
    ax0 = ox + (R + 30) * sx
    base = oy                       # axis line
    t_c, b, w = comp.carrier["plate_t"], g.sun.b, 1.0
    cl = 0.5
    z = ax0
    def rect(x, r0, r1, length, fill):
        return (f'<rect x="{x:.2f}" y="{base - r1 * sx:.2f}" width="{length * sx:.2f}" height="{(r1 - r0) * sx:.2f}" '
                f'fill="{fill}" stroke="#222" stroke-width="0.6"/>')
    x_pl1 = z + cl * sx
    x_g = x_pl1 + (t_c + w) * sx
    x_pl2 = x_g + (b + w) * sx
    out.append(rect(z, g.ring.df / 2, R, L, "#e8eef5"))                                       # ring body
    out.append(rect(x_g, g.ring.da / 2, g.ring.df / 2, b, "#cfd9e6"))                          # ring teeth
    out.append(rect(x_pl1, g.sun.da / 2 + 1, comp.carrier["plate_d"] / 2, t_c, "#cfe8d6"))     # plates
    if d.req.carrier != "single":
        out.append(rect(x_pl2, g.sun.da / 2 + 1, comp.carrier["plate_d"] / 2, t_c, "#cfe8d6"))
    out.append(rect(x_g, g.a_w - g.planet.da / 2, g.a_w + g.planet.da / 2, b, "#f6e7c8"))     # planet
    out.append(rect(x_pl1, g.a_w - comp.pin["d"] / 2, g.a_w + comp.pin["d"] / 2,
                    (1 if d.req.carrier == "single" else 2) * t_c + b + 2 * w, "#bbb"))   # pin
    out.append(rect(x_g + 0.5 * sx, g.a_w + comp.pin["d"] / 2, g.a_w + comp.bearing["Ew"] / 2, comp.bearing["Bc"], "#999"))  # needles
    out.append(rect(x_g, d.req.shaft_d / 2, g.sun.da / 2, b, "#d8e8d0"))                       # sun
    out.append(rect(z - 10 * sx, 0, d.req.shaft_d / 2, L + 20, "#ddd"))                        # shaft
    out.append(f'<line x1="{z - 15 * sx:.1f}" y1="{base:.1f}" x2="{z + (L + 15) * sx:.1f}" y2="{base:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="{z + L / 2 * sx:.1f}" y="{H + 15:.0f}" text-anchor="middle">Axial half-section (input at left)</text>')
    # data block
    tx = z + (L + 25) * sx
    lines = [f"i = {g.ratio:.4f}  ({g.sun.z}/{g.planet.z}/{g.ring.z}, {g.n_planets} planets)",
             f"module {g.sun.m:g} mm, α = 20°, face {b:g} mm",
             f"a_w = {g.a_w:.4f} mm (pin circle Ø{2 * g.a_w:.3f})",
             f"ring Ø{2 * R:g} × {L:.1f} mm stack",
             f"sun on Ø{d.req.shaft_d:g} shaft, {d.components.sun_hub.get('key', d.req.hub_joint)}",
             f"planet bearing {comp.bearing['type'].split(' (')[0].replace('needle roller cage assembly ', '')}",
             f"pin Ø{comp.pin['d']:g} j5 × {comp.pin['length']:g}",
             f"carrier plates Ø{comp.carrier['plate_d']:g} × {t_c:g}",
             f"T_out {d.req.torque_out:g} N·m at {d.req.n_in / g.ratio:.0f} rpm",
             f"S_H {d.rating.S_H:.2f} / S_F {d.rating.S_F:.2f}",
             f"L10h planet bearing ≈ {comp.bearing['L10h_est']:.3g} h"]
    for i, s in enumerate(lines):
        out.append(f'<text x="{tx:.1f}" y="{30 + 16 * i}">{s}</text>')
    out.append("</svg>")
    return "\n".join(out)

## Report and Package: Theory

The package collects the results of the earlier notebooks; the few values computed here are:

- **Planet data sheet.** The planet meshes with both the sun and the ring, so its active profile starts at the lower of the two: $d_{Nf,p} = \min(d_{Nf2,sp},\ d_{Nf1,pr})$.
- **Carrier pin holes.** The pin-hole position is held to the centre distance tolerance: a position tolerance zone of diameter $2f_a = IT7(a_w)$ about the nominal pin circle $2a_w$, and the angular spacing $360°/N$ to within $\pm\,(f_a / a_w)$ rad, written in arc minutes, $\pm\,60\cdot\frac{180}{\pi}\frac{f_a}{a_w}$′. Both plates are line-bored together.
- **Fits named on the parts:** planet bore G6 (needle outer raceway), pin j5, pin holes N7, the sun joint (keyed: bore H7 on a k6 shaft, key groove P9; spline: DIN 5480 fit; press: the chosen H6/r5 … H7/u6 fit; adhesive: H7/k6; integral: no fit), carrier plate OD h11, ring OD h7 when bolted or s6 when pressed into an H7 housing. Their limits come from `gears.manufacturing.iso286_limits`.
- **DXF outlines** are written at the mean thickness $\Delta s = (A_{sne} + A_{sni})/2$, with the bore (sun: $d_{sh}$, planet: $E_w$) or the ring outside diameter as a circle.

In [ ]:
#| export
def _fmt(v) -> str:
    if isinstance(v, float):
        return f"{v:.4g}"
    return str(v)


def component_rows(d: PlanetaryDesign) -> list[tuple[str, str, str]]:
    "(part, feature, specification) rows with numeric limits for the part drawings."
    c, g = d.components, d.gears
    Fw, Ew = c.bearing["Fw"], c.bearing["Ew"]
    rows = [("planet", "bore (bearing outer raceway)", fmt_fit(Ew, "G6") + ", 58–64 HRC, Ra ≤ 0.2 µm, cylindricity 0.003"),
            ("planet", "faces", f"run-out to bore ≤ 0.01 mm, width {g.planet.b:g} −0.05/−0.10 mm"),
            ("planet pin", "diameter", fmt_fit(Fw, "j5") + ", 58–64 HRC, Ra ≤ 0.2 µm"),
            ("planet pin", "length", f"{c.pin['length']:g} mm, chamfer 0.5 × 45° both ends"),
            ("carrier plates", "pin holes", f"{g.n_planets} × " + fmt_fit(Fw, "N7") + f" on Ø{2 * g.a_w:.3f} pin circle"),
            ("carrier plates", "pin hole position", f"position Ø{2 * it_grade(g.a_w, 7) / 2:.3f} mm to bore axis (= ±f_a on centre distance); "
                                                   f"spacing {360 / g.n_planets:g}° ±{math.degrees(it_grade(g.a_w, 7) / 2 / g.a_w) * 60:.1f}′; "
                                                   + ("both plates line-bored together" if d.req.carrier != "single" else
                                                      "holes square to the plate face within 0.01 mm over the seat")),
            ("carrier plates", "outside diameter", f"Ø{c.carrier['plate_d']:g} h11, thickness {c.carrier['plate_t']:g} mm"),
            ("carrier plates", "centre bore (sun clearance)", f"Ø{math.ceil(c.carrier['sun_clearance_bore']):g} mm min"),
            ("carrier", "posts", f"{g.n_planets} posts {c.carrier['post_width']:.1f} wide × {c.carrier['post_height']:.1f} radial × {c.carrier['post_length']:.1f} long"
             if "post_width" in c.carrier else "none: single plate, pins cantilevered; " + c.pin["fit_in_plates"]),
            ("carrier", "output", f"output shaft Ø{math.ceil(c.carrier['output_shaft_min_d']):g} mm min (torsion), bearing seats per housing design"),
            ("thrust washer", "size", c.axial["thrust_washers"]),
            ]
    hub = c.sun_hub
    if hub["joint"] == "press":
        d_sh = hub["shaft_d"]
        h_, s_ = hub["press_fit"].split("/")
        rows += [("sun", "bore", fmt_fit(d_sh, h_) + f", Rz ≤ 2 µm (ground); interference fit {hub['press_fit']}"),
                 ("input shaft", "sun seat", fmt_fit(d_sh, s_) + f", Rz ≤ 2 µm, length ≥ {g.sun.b:g} mm"),
                 ("sun", "assembly", hub["assembly"])]
    elif hub["joint"] == "adhesive":
        d_sh = hub["shaft_d"]
        rows += [("sun", "bore", fmt_fit(d_sh, "H7") + ", bonded with anaerobic retaining compound"),
                 ("input shaft", "sun seat", fmt_fit(d_sh, "k6") + ", degreased and activated before bonding")]
    elif hub["joint"] == "integral":
        rows += [("sun / input shaft", "teeth", "sun teeth cut and ground on the input shaft; " + hub["note"]),
                 ("sun / input shaft", "hob run-out", f"{hub['hob_runout']:.1f} mm from the gear face to the next shoulder"
                  if hub["hob_runout"] else "shaft beside the sun below the root circle")]
    elif hub["joint"] == "spline":
        rows.append(("sun", "bore", hub["spline"] + " (" + hub["backlash"] + ")"))
    if hub["joint"] == "key":
        d_sh = hub["shaft_d"]
        rows += [("sun", "bore", fmt_fit(d_sh, "H7") + " (shaft k6)"),
                 ("sun", "key groove", f"width {hub['key_b']:g} P9 {_dev(hub['key_b'], 'P9')}, depth {d_sh + hub['t2']:.1f} +0.1 over bore side"),
                 ("key", "parallel key", hub["key"] + ", C45+C"),
                 ("input shaft", "key seat", f"width {hub['key_b']:g} P9, depth t1 = {hub['t1']:g} +0.1 mm")]
    rm = c.ring_mount
    rows.append(("ring", "outside diameter", f"Ø{rm['ring_od']:g} " + ("h7" if rm["bolts"] else "s6 (press into housing H7)")))
    rows.append(("ring", "fixing", rm["method"] + (f", tightening torque {rm['tightening_torque']:g} N·m" if rm["bolts"] else "")))
    return rows


def _dev(dn: float, fit: str) -> str:
    up, lo = iso286_limits(dn, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def data_sheets(d: PlanetaryDesign) -> dict[str, list[DataRow]]:
    "Gear data tables of sun, planet and ring."
    g, al = d.gears, d.allowances["allowances"]
    mats = d.req.material_set()
    fin = d.req.flank_finish
    return {
        "sun": gear_data_sheet(g.sun, g.planet, g.a_w, al["sun"], d.req.grade, mats["sun"], fin,
                               f" ({g.n_planets} planets)", d_active=g.sun_planet.d_Nf1),
        "planet": gear_data_sheet(g.planet, g.sun, g.a_w, al["planet"], d.req.grade, mats["planet"], fin,
                                  f", and ring z = {g.ring.z}", d_active=min(g.sun_planet.d_Nf2, g.planet_ring.d_Nf1)),
        "ring": gear_data_sheet(g.ring, g.planet, g.a_w, al["ring"], d.req.ring_grade, mats["ring"], "shaped",
                                f" ({g.n_planets} planets)", d_active=g.planet_ring.d_Nf2),
    }


def parts_list(d: PlanetaryDesign) -> list[tuple[int, str, int, str, str]]:
    "(item, part, qty, description, material/standard)."
    g, c = d.gears, d.components
    mats = d.req.material_set()
    N = g.n_planets
    sun_desc = (f"z {g.sun.z}, m {g.sun.m:g}, b {g.sun.b:g}, cut on the input shaft (shaft Ø{d.req.shaft_d:g} beside it)"
                if c.sun_hub["joint"] == "integral" else
                f"z {g.sun.z}, m {g.sun.m:g}, b {g.sun.b:g}, bore Ø{d.req.shaft_d:g} {c.sun_hub['bore_fit']}")
    items = [("Sun gear" if c.sun_hub["joint"] != "integral" else "Sun pinion shaft", 1, sun_desc, mats["sun"].name + ", " + mats["sun"].treatment),
             ("Planet gear", N, f"z {g.planet.z}, m {g.planet.m:g}, b {g.planet.b:g}, bore Ø{c.bearing['Ew']:g} G6", mats["planet"].name + ", " + mats["planet"].treatment),
             ("Ring gear", 1, f"z {g.ring.z}, m {g.ring.m:g}, b {g.ring.b:g}, OD Ø{c.ring_mount['ring_od']:g}", mats["ring"].name + ", " + mats["ring"].treatment),
             ("Planet bearing", N, c.bearing["type"], f"C_req {c.bearing['C_req'] / 1000:.2f} kN"),
             ("Planet pin", N, f"Ø{c.pin['d']:g} j5 × {c.pin['length']:g}", c.pin["material"]),
             ("Thrust washer", 2 * N, c.axial["thrust_washers"], ""),
             ]
    if "post_width" in c.carrier:
        items += [("Carrier, output plate with posts", 1, f"Ø{c.carrier['plate_d']:g} × {c.carrier['plate_t']:g} + {N} posts", c.carrier["material"]),
                  ("Carrier, free plate", 1, f"Ø{c.carrier['plate_d']:g} × {c.carrier['plate_t']:g}, screwed and doweled to the posts", c.carrier["material"])]
    else:
        items += [("Carrier plate (output)", 1, f"Ø{c.carrier['plate_d']:g} × {c.carrier['plate_t']:g}, {N} pins pressed in", c.carrier["material"]),
                  ("Retaining ring for pin", N, f"DIN 471 for Ø{c.pin['d']:g} shaft", "spring steel")]
    if c.sun_hub["joint"] == "key":
        items.append(("Parallel key", 1, c.sun_hub["key"], "C45+C, DIN 6885-1"))
    if c.ring_mount["bolts"]:
        items.append(("Ring clamping bolt", c.ring_mount["bolts"], f"{d.components.ring_mount['method'].split(' through')[0].split('× ')[1]} socket head, ISO 4762", "8.8"))
        items.append(("Dowel pin", 2, "ISO 8734, locate ring", "hardened"))
    else:
        items.append(("Dowel pin", 2, "ISO 8734, secure pressed ring against turning", "hardened"))
    return [(i + 1, *row) for i, row in enumerate(items)]


def report_markdown(d: PlanetaryDesign) -> str:
    "The design report (see the package table above)."
    r, g, c, req = d.rating, d.gears, d.components, d.req
    sp = g.speeds(req.n_in)
    L = [f"# Single-stage planetary gear: design report", "",
         "Ring gear fixed, sun input, carrier output. Spur gears, 20° pressure angle, ISO 53 profile A. "
         "Lengths in mm, torques in N·m, stresses in MPa. Calculated with `emachines.gears`.", "",
         "## Requirements", "",
         "| Item | Value |", "|---|---|"]
    for k, v in asdict(req).items():
        L.append(f"| {k} | {v} |")
    L += ["", "## Gear set", "", "```", g.summary(), "```", "",
          f"Speeds at {req.n_in:g} rpm input: carrier {sp['carrier']:.1f} rpm, planets {sp['planet_rel']:.1f} rpm on their pins "
          f"({sp['planet_abs']:.1f} rpm absolute). Mesh frequency {g.mesh_frequency(req.n_in):.0f} Hz.", "",
          f"Torques: sun {r.T_sun:.3f}, carrier {r.T_carrier:.3f}, ring reaction {r.T_ring:.3f}. "
          f"Tangential force per planet mesh {r.F_t:.1f} N (before K_γ). Pitch line velocity {r.v:.2f} m/s.", "",
          "## Load capacity (ISO 6336)", "", "Factors: " + ", ".join(f"{k} = {v:.3f}" for k, v in r.factors.items()), "",
          "```", r.table(), "```", "",
          "## Backlash and tooth thickness", "",
          f"Required minimum normal backlash {d.allowances['j_bn_min_required']:.3f} mm; centre distance tolerance ±{d.allowances['f_a'] * 1000:.0f} µm.", "",
          "| Gear | A_sne | A_sni |", "|---|---|---|"]
    for k, a in d.allowances["allowances"].items():
        L.append(f"| {k} | {a.A_sne:+.3f} | {a.A_sni:+.3f} |")
    for mesh, (jmin, jmax) in d.allowances["backlash_n"].items():
        L.append(f"\n{mesh}: normal backlash {jmin:.3f} to {jmax:.3f} mm.")
    for name, rows in data_sheets(d).items():
        L += ["", f"## Gear data — {name}", "", "| Item | Symbol | Value |", "|---|---|---|"]
        L += [f"| {x.item} | {x.symbol} | {x.value} |" for x in rows]
    L += ["", "## Components", ""]
    L += ["| Part | Feature | Specification |", "|---|---|---|"] + [f"| {a} | {b_} | {s} |" for a, b_, s in component_rows(d)]
    L += ["", "### Planet bearing", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.bearing.items()]
    L += ["", "### Planet pin", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.pin.items()]
    L += ["", "### Carrier", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.carrier.items()]
    L += ["", "### Sun hub", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.sun_hub.items()]
    L += ["", "### Ring fixing", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.ring_mount.items()]
    L += ["", "### Axial stack", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.axial.items()]
    L += ["", "## Checks", ""] + [f"- {ch}" for ch in c.checks]
    L += [f"- {'✓' if r.ok else '✗'} gear rating: S_H {r.S_H:.2f} ≥ {req.S_Hmin}, S_F {r.S_F:.2f} ≥ {req.S_Fmin}"]
    L += ["", "## Lubrication", "", c.lubrication, "",
          "## Parts list", "", "| # | Part | Qty | Description | Material / standard |", "|---|---|---|---|---|"]
    L += [f"| {i} | {p} | {q} | {ds} | {mt} |" for i, p, q, ds, mt in parts_list(d)]
    L += ["", "## Assumptions to confirm before release", "",
          "- Material allowables are ISO 6336-5 MQ typical values; confirm with the steel certificate and heat treater.",
          (f"- K_Hβ = {req.rating_inputs().K_Hbeta} assumes a stiff double-plate carrier and crowned or ground planets." if req.carrier != "single" else
           f"- K_Hβ = {req.rating_inputs().K_Hbeta}; the cantilevered pins tilt the planets (K_Hβ from the pin slope "
           f"{c.pin.get('K_Hbeta_from_slope', 1):.2f}); crown the planets or add a lead correction."),
          "- The planet bearing rating is an ISO 281 estimate; select the catalogue part with C_r ≥ C_req and check its speed limit.",
          "- Tooth thickness tolerances follow DIN 3967 practice; agree them with the gear shop.",
          "- The ring's root form diameter is estimated from a rack; confirm with the shaper cutter data.",
          "- Re-check the final design in a full ISO 6336 tool if any safety factor is within 10 % of its minimum."]
    return "\n".join(L) + "\n"


def write_package(d: PlanetaryDesign, folder: str) -> list[str]:
    "Write the manufacturing package (see the table above) into `folder`; returns the file paths."
    os.makedirs(folder, exist_ok=True)
    paths = []
    def p(name):
        paths.append(os.path.join(folder, name))
        return paths[-1]
    with open(p("report.md"), "w") as f:
        f.write(report_markdown(d))
    for name, rows in data_sheets(d).items():
        with open(p(f"gear_data_{name}.csv"), "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["item", "symbol", "value"])
            w.writerows([(x.item, x.symbol, x.value) for x in rows])
    with open(p("parts.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["item", "part", "qty", "description", "material_or_standard"])
        w.writerows(parts_list(d))
    with open(p("component_dimensions.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["part", "feature", "specification"])
        w.writerows(component_rows(d))
    with open(p("layout.svg"), "w") as f:
        f.write(layout_svg(d))
    al = d.allowances["allowances"]
    g = d.gears
    write_dxf(p("sun.dxf"), [gear_outline(g.sun, (al["sun"].A_sne + al["sun"].A_sni) / 2)],
              [] if d.req.hub_joint == "integral" else [(0, 0, d.req.shaft_d / 2)])
    write_dxf(p("planet.dxf"), [gear_outline(g.planet, (al["planet"].A_sne + al["planet"].A_sni) / 2)],
              [(0, 0, d.components.bearing["Ew"] / 2)])
    write_dxf(p("ring.dxf"), [gear_outline(g.ring, (al["ring"].A_sne + al["ring"].A_sni) / 2)],
              [(0, 0, d.components.ring_mount["ring_od"] / 2)])
    return paths


PlanetaryDesign.report = report_markdown            # type: ignore[attr-defined]
PlanetaryDesign.write_package = write_package       # type: ignore[attr-defined]
PlanetaryDesign.svg = layout_svg                    # type: ignore[attr-defined]

## Worked Example

**These are placeholder inputs**, chosen to show the method; replace them with the real envelope and shaft. A 120 mm diameter, 40 mm long envelope, a 14 mm keyed motor shaft, ratio 5, 20 N·m at the output, 3000 rpm in, 10 000 h:

In [ ]:
import time

In [ ]:
req = PlanetaryRequirements(ratio=5.0, torque_out=20.0, n_in=3000.0, envelope_d=120.0, envelope_length=40.0,
                            shaft_d=14.0, life_h=10_000)
t0 = time.time()
designs = design_planetary(req)
print(f"{len(designs)} designs in {time.time() - t0:.1f} s")
for dd in designs:
    print(" ", dd.label())
best = designs[0]
assert best.rating.ok and best.components.ok and abs(best.ratio_error) <= req.ratio_tol

The chosen design, with all component checks:

In [ ]:
print(best.gears.summary())
print(best.rating.table())
for ch in best.components.checks:
    print(ch)

Its layout:

In [ ]:
from IPython.display import SVG, display

In [ ]:
display(SVG(best.svg()))

And the manufacturing package written to a folder:

In [ ]:
import tempfile

In [ ]:
with tempfile.TemporaryDirectory() as td:
    files = write_package(best, td)
    for f_ in files:
        print(os.path.basename(f_), os.path.getsize(f_), "bytes")
    print(open(os.path.join(td, "gear_data_planet.csv")).read()[:1500])

---

## Tests

In [ ]:
#| hide
import xml.dom.minidom

In [ ]:
#| hide
# every design returned passes everything, and is sorted
for dd in designs:
    assert dd.rating.ok and dd.components.ok and dd.gears.ok
    assert dd.length <= req.envelope_length + 1e-9
keys = [(round(abs(x.ratio_error) / 0.005), round(x.length), -x.margin) for x in designs]
assert keys == sorted(keys)
# more torque never gives a shorter best design
req2 = replace(req, torque_out=60.0)
d2 = design_planetary(req2, max_results=1)
assert not d2 or d2[0].length >= best.length - 1e-9
# impossible envelope returns nothing rather than a failing design
assert design_planetary(replace(req, envelope_d=40.0)) == []
# every sun joint gives a design, and the joints without backlash refuse a floating sun
for j, opt in (("spline", {}), ("press", {}), ("adhesive", {}), ("integral", {}), ("spline", {"spline_fit": "tight"})):
    dj = design_planetary(replace(req, hub_joint=j, joint_options=opt), max_results=1)
    assert dj and dj[0].components.ok, j
    assert component_rows(dj[0]) and parts_list(dj[0])
# a single-plate carrier reaches ratio 9 in a Ø70 ring, where posts do not fit
r9 = PlanetaryRequirements(ratio=9.0, torque_out=9.0, n_in=3000.0, envelope_d=70.0, shaft_d=6.0,
                           hub_joint="integral", carrier="single", m_min=0.5)
d9 = design_planetary(r9, max_results=1)
assert d9 and d9[0].components.ok and abs(d9[0].ratio_error) < 0.01 and d9[0].gears.sun.m >= 0.5
assert design_planetary(replace(r9, carrier="double"), max_results=1) == []
assert "post_width" not in d9[0].components.carrier and parts_list(d9[0])
try:
    design_planetary(replace(req, hub_joint="press", floating_sun=True))
    raise AssertionError("floating sun with a press fit should be refused")
except ValueError:
    pass
# the SVG is well formed XML
xml.dom.minidom.parseString(best.svg())
# report mentions every check
rep = best.report()
for ch in best.components.checks:
    assert ch.name in rep